# PromptSum Kaggle Runner
This notebook runs PromptSum directly from the git repo in Kaggle.

It is configured for a small, practical run (few-shot subset) so you can validate the pipeline first.

In [ ]:
# Basic setup
import os
from pathlib import Path

# Set this to your fork URL so Kaggle gets your latest repo fixes.
REPO_URL = "https://github.com/<your-username>/PromptSum.git"
REPO_REF = "main"  # branch, tag, or commit SHA
FORCE_RECLONE = True

REPO_DIR = Path('/kaggle/working/PromptSum')
HF_CACHE = Path('/kaggle/working/hf_cache')

print('Repo URL:', REPO_URL)
print('Repo ref:', REPO_REF)
print('Repo dir:', REPO_DIR)
print('HF cache:', HF_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')
HF_CACHE.mkdir(parents=True, exist_ok=True)

In [ ]:
# Clone repo and pin exact ref (avoids stale workspace state)
import shutil
import subprocess

if REPO_DIR.exists() and FORCE_RECLONE:
    shutil.rmtree(REPO_DIR)
    print('Removed existing repo because FORCE_RECLONE=True')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

%cd /kaggle/working/PromptSum
!git rev-parse --short HEAD
!git remote -v

In [ ]:
# Install a compatible dependency set for the repo on Kaggle
# Note: We avoid pinning torch here to keep Kaggle's preinstalled CUDA-enabled torch.
# !python -m pip install -q --upgrade pip setuptools wheel
# !python -m pip uninstall -y spacy thinc catalogue srsly blis cymem preshed murmurhash || true

# !python -m pip install -q \
#   transformers==4.31.0 \
#   datasets==2.13.2 \
#   huggingface_hub==0.16.4 \
#   fsspec==2023.1.0 \
#   sentencepiece==0.1.99 \
#   scipy==1.10.1 \
#   rouge-score==0.1.2 \
#   bert-score==0.3.13 \
#   fairscale==0.4.13 \
#   importlib-metadata==4.13.0 \
#   spacy==3.5.4 \
#   catalogue==2.0.10
# !python -m spacy download en_core_web_sm -q

# !python - <<'PY'
import sys, torch, transformers, datasets, huggingface_hub, fsspec, spacy, catalogue
print(sys.version)
print('torch', torch.__version__)
print('cuda?', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
print('hub', huggingface_hub.__version__)
print('fsspec', fsspec.__version__)
print('spacy', spacy.__version__)
print('catalogue', getattr(catalogue, '__version__', 'unknown'))
# PY

In [ ]:
# Patch only hardcoded local paths in src/hyperparameters.py for Kaggle
from pathlib import Path

hp_path = Path('src/hyperparameters.py')
text = hp_path.read_text(encoding='utf-8')

new_root = "root = '/kaggle/working/PromptSum/' # local path to the the directory where PromptSum is cloned"
new_cache = "cache_path = '/kaggle/working/hf_cache/transformers/' # local path to cache directory with the backbone's model weights"

lines = text.splitlines()
for i, line in enumerate(lines):
    if line.startswith('root = '):
        lines[i] = new_root
    if line.startswith('cache_path = '):
        lines[i] = new_cache

hp_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Patched', hp_path)

# Sanity checks for repo-level fixes you pushed
utils_text = Path('src/utils.py').read_text(encoding='utf-8')
print('Has SAMSum loader mapping:', 'knkarthick/samsum' in utils_text)
assert 'knkarthick/samsum' in utils_text, 'Missing SAMSum mapping in src/utils.py. Did you clone the right repo/ref?'

ds_text = Path('src/dataset/dataset.py').read_text(encoding='utf-8')
print('Uses verification_mode=no_checks:', 'verification_mode="no_checks"' in ds_text)

: 

In [ ]:
# Optional: show GPU and free disk before run
!nvidia-smi || true
!df -h /kaggle/working || true

## Run PromptSum on a subset
This command uses a very small subset so it finishes quickly and avoids the high-memory entity-tagger path.

In [ ]:
# IMPORTANT: --use_tagger/--use_entity_chain/--infer_val_entities are store_false flags in this repo.
# Passing them disables those components and reduces RAM usage.
!mkdir -p log
!python src/main_few_shot.py \
  --model PegasusFinetune \
  --dataset_name samsum \
  --few_shot 5 \
  --num_seeds 1 \
  --seeds_to_keep 0 \
  --finetune_summary \
  --max_test_size 20 \
  --log_dir ./log \
  --log_name kaggle_run \
  --use_tagger \
  --use_entity_chain \
  --infer_val_entities \
  --use_pretrain_ckpt \
  --cache_path /kaggle/working/hf_cache/transformers/

## Scale up after sanity check
Once this works, increase `--few_shot` and `--max_test_size`, or switch model variants/scripts as needed.